# MFA Align
Данный ноутбук посвящен процессу выравнивания набора данных RUSLAN с помощью инструмента Montreal Forced Aligner.

Модели, используемые для выравнивания:
* [russian acoustic model v3.1.0](https://mfa-models.readthedocs.io/en/latest/acoustic/Russian/Russian%20MFA%20acoustic%20model%20v3_1_0.html)
* [russian g2p model v3.1.0](https://mfa-models.readthedocs.io/en/latest/g2p/Russian/Russian%20MFA%20G2P%20model%20v3_1_0.html)
* [russian dictionary v3.1.0](mfa-models.readthedocs.io/en/latest/dictionary/Russian/Russian%20MFA%20dictionary%20v3_1_0.html)  
Также доступны на [huggingface](https://huggingface.co/MontrealCorpusTools/russian_mfa)

In [1]:
import pandas as pd
import json
import glob
import numpy as np
import os

#### Константы: пути к данным и моделям
* RUSLAN_DIR -- место хранения исходных данных -- файла метаданных и аудио
* TEXTGRID_DIR -- место, куда будут сохранены текстовки с разметкой
* MFA_MODELS_DIR -- место для сохранения предобученных и потенциально обученных/доработанных моделей и артефактов от MFA

In [2]:
RUSLAN_DIR = '../../data'
RUSLAN_DATA = '../../data/RUSLAN' # здесь лежат wav + их txt
TEXTGRID_DIR = '../../data/RUSLAN_align_byPelse_attempt_2_v1' # сюда положим результат
MFA_MODELS_DIR = '../../models/mfa'

#### Чтение списка файлов

Файл должен быть результатом нормализации: То есть  в нём помимо двух колонок (идентификатор, сырой текст), должен быть также и нормализованный текст.

In [3]:
import csv
ruslan = pd.read_csv(f'{RUSLAN_DIR}/metadata_RUSLAN_22200_normalized_byPelse.csv', 
                     sep='|', 
                     names=['id', 'raw', 'nrm'], 
                     quoting=csv.QUOTE_NONE) # Игнорировать кавычки иначе количество строк будет расходиться с csv
print(len(ruslan)) # должно быть 21328+-

21328


In [29]:
#ruslan.head(3)
ruslan.iloc[6685:6687]

,id,raw,nrm
6685,006917_RUSLAN,"Жбанков вдруг напрягся и говорит Пардон, я это...","Жбанков вдруг напрягся и говорит Пардон, я это..."
6686,006918_RUSLAN,"КВВК, поправила Белла.","ка-вэ-вэ-ка, поправила Белла."


#### Сохранение текстовок для выравнивателя

Необходимо положить файл с расширением .txt рядом с каждым аудифайлом

In [33]:
for i, t in ruslan[['id', 'nrm']].values:
    open(os.path.join(f'{RUSLAN_DIR}/RUSLAN', i+'.txt'), 'w', encoding='utf-8').write(t)

#### Настройка среды MFA
Предпочтительно через докер, но можно его собрать и в системе

`docker pull mmcauliffe/montreal-forced-aligner:v3.4.1`  
`docker run -it -v $RUSLAN_DIR:/data -v $TEXTGRID_DIR:/align -v $MFA_MODELS_DIR:/models mmcauliffe/montreal-forced-aligner:v3.4.1`  
`mfa model download g2p russian_mfa`  
`mfa model download acoustic russian_mfa`  
`mfa model download dictionary russian_mfa`  
  
`mfa align /data /models/russian_mfa.dict /models/russian_mfa_acoustic /align/v1`

**Важно!** Скачивание моделей через `mfa model` в версии `3.4.1` является устаревшим и планируется к удалению. Можно скачать модели с hugginface или напрямую с сайта. 

Результат работы команды mfa align -- директория с файлами разметки .TextGrid.

#### Выполняю в powershell с установленной средой для mfa

1) Так как это не первая попытка, нужно добавить флаг *--clean* (или -c) в самый конец команды.

mfa align ../../data/RUSLAN ../../models/mfa/russian_mfa.dict ../../models/mfa/russian_mfa_acoustic.zip ../../data/RUSLAN_align_byPelse_attempt_2_v1 --clean


#### Проверка результата: потребуется библиотека praatio для работы с TextGrid

In [4]:
from praatio import textgrid
from prepare_training_data import read_text_grids

In [5]:
ruslan['phn'] = read_text_grids(ruslan, TEXTGRID_DIR)[2]

100%|███████████████████████████████████████████████████████████████████████████| 21328/21328 [00:35<00:00, 606.88it/s]


### Анализ ошибок разметки: поиск Out-Of-Vocabulary слов
Чтение словаря и анализ корпуса на предмет OOV-слов.

Метка `spn` на уровне фонем, а.к. `spoken noise` -- фактор, говорящий о том, что слово не разметилось; одна из причин -- его нет в словаре.

In [9]:
ru_dict = pd.read_csv(f'{MFA_MODELS_DIR}/russian_mfa.dict', names=['grapheme', 'p1', 'p2', 'p3', 'p4', 'phoneme'], sep='\t')
ru_dict.loc[ru_dict.phoneme.isna(), 'phoneme'] = ru_dict[ru_dict.phoneme.isna()]['p1']

words = ' '.join(ruslan[ruslan.phn.str.contains('spn')].raw.str.lower().str.replace('[^а-яё ]','', regex=True)).split(' ')
print("Всего слов с ошибками:", len(words), "| Уникальных:", len(set(words)))

dict_words = set(ru_dict.grapheme.unique())

pd.DataFrame({'txt': list(set(words)-dict_words)}).to_csv(f'{RUSLAN_DIR}/oovs_found.txt', index=False, header=False)

Всего слов с ошибками: 137959 | Уникальных: 34579


#### Предсказание вариантов произношения для OOV-слов с помощью G2P-модели 

`mfa g2p oovs_found.txt /models/russian_mfa_g2p oovs_mfa_g2p_3_1_0.txt`

mfa g2p [ЧТО_ПЕРЕВОДИМ] [МОДЕЛЬ_G2P] [КУДА_СОХРАНЯЕМ]

1. oovs_found.txt (Входной файл со словами)
	• Что это: Это текстовый файл, который создан на предыдущем шаге. В нём лежит простой список слов (каждое с новой строки), для которых MFA выдал метку spn.
	• Что передавать: Путь к этому файлу.

2. /models/russian_mfa_g2p (G2P-модель)
	• Что это: Это обученная модель, которая знает правила чтения букв как звуков для русского языка. Обычно это файл с расширением .zip (иногда в командах расширение опускают, если MFA скачал её во внутреннюю папку).
	• Что передавать: Путь к файлу модели russian_mfa_g2p.zip.

3. oovs_mfa_g2p_3_1_0.txt (Выходной файл-результат)
	• Что это: Это файл, который создастся сам в результате работы команды. Он будет выглядеть как новый кусочек словаря: слово  ф о н е м ы.
	• Что передавать: Путь, куда сохраняем с названием файла

#### Запускаю в powershell

mfa g2p ../../data/oovs_found.txt ../../models/mfa/russian_mfa_g2p.zip ../../data/oovs_mfa_g2p_3_1_0.txt

In [10]:
with open(f'{MFA_MODELS_DIR}/russian_mfa_oov_ruslan.dict', 'w') as writer:
    dict_orig =open(f'{MFA_MODELS_DIR}/russian_mfa.dict').read().strip().split('\n')
    oov_trans = open(f'{RUSLAN_DIR}/oovs_mfa_g2p_3_1_0.txt').read().strip().split('\n')
    writer.write('\n'.join(sorted(dict_orig+oov_trans)))

#### Повторный запуск выравнивания

После пополнения списка OOV-словами, выравнивание должно пойти лучше.

#### В powershell выполняю 
`mfa align ../../data/RUSLAN ../../models/mfa/russian_mfa_oov_ruslan.dict ../../models/mfa/russian_mfa_acoustic.zip ../../data/RUSLAN_align_byPelse_attempt_2_v2`

In [12]:
%run prepare_training_data.py

100%|████████████████████████████████████████████████████████████████████████████| 21328/21328 [06:21<00:00, 55.94it/s]


In [23]:
# Загружаем датасет с правильным сепаратором и игнорированием кавычек
preview_df = pd.read_csv('../../data/RUSLAN_pause_metadata_byPelse.csv', sep='|', quoting=csv.QUOTE_NONE)

# preview_df.head(10)
preview_df.iloc[239:253]

,label,duration,id,label_raw,is_last_word,is_pause_after,pause_duration,set
239,обладаем,0.580000,000033_RUSLAN,обладаем,0,0,0.00,train
240,равными,0.580000,000033_RUSLAN,равными,0,1,0.10,train
241,авторскими,0.640000,000033_RUSLAN,авторскими,0,1,0.07,train
242,правами,0.335918,000033_RUSLAN,правами.,1,0,0.00,train
243,обязанности,0.840000,000034_RUSLAN,Обязанности,0,1,0.03,train
244,разные,0.557755,000034_RUSLAN,разные.,1,0,0.00,train
245,как,0.190000,000035_RUSLAN,Как,0,0,0.00,test
246,минимум,0.610000,000035_RUSLAN,"минимум,",0,1,0.12,test
247,одним,0.570000,000035_RUSLAN,одним,0,1,0.03,test
248,неотъемлемым,0.940000,000035_RUSLAN,неотъемлемым,0,1,0.12,test


#### Attempt 1
Calculate metrics, training fold; Exclude last tokens in every sentence!
PRC: 0.3946, REC: 0.3917, F1: 0.3932; MAE: 0.1297;

Calculate metrics, testing fold; Exclude last tokens in every sentence!
PRC: 0.3881, REC: 0.3921, F1: 0.3901; MAE: 0.1246;

#### Attempt 2, Поставили паузы с учетом всех знаков препинания, а не только запятой
Calculate metrics, training fold; Exclude last tokens in every sentence!
PRC: 0.4988, REC: 0.6642, F1: 0.5698; MAE: 0.1257;

Calculate metrics, testing fold; Exclude last tokens in every sentence!
PRC: 0.4938, REC: 0.6616, F1: 0.5655; MAE: 0.1225;

#### Attempt 3, Поверяемая длина слова было 10, стало 11
Calculate metrics, training fold; Exclude last tokens in every sentence!
PRC: 0.5421, REC: 0.6334, F1: 0.5842; MAE: 0.1263;

Calculate metrics, testing fold; Exclude last tokens in every sentence!
PRC: 0.5393, REC: 0.6319, F1: 0.5819; MAE: 0.1230;

#### Attempt 4, Поверяемая длина слова было 11, стало 12
Calculate metrics, training fold; Exclude last tokens in every sentence!
PRC: 0.5767, REC: 0.6113, F1: 0.5935; MAE: 0.1266;

Calculate metrics, testing fold; Exclude last tokens in every sentence!
PRC: 0.5750, REC: 0.6106, F1: 0.5923; MAE: 0.1235;

#### Attempt 5, обрабатываем дефис отдельно, добавили правилодля слова "чтобы"
Calculate metrics, training fold; Exclude last tokens in every sentence!
PRC: 0.6046, REC: 0.6096, F1: 0.6070; MAE: 0.1266;

Calculate metrics, testing fold; Exclude last tokens in every sentence!
PRC: 0.6002, REC: 0.6084, F1: 0.6043; MAE: 0.1237;

#### Attempt 6, пауза после слова "говорит"
Calculate metrics, training fold; Exclude last tokens in every sentence!
PRC: 0.6044, REC: 0.6100, F1: 0.6072; MAE: 0.1266;

Calculate metrics, testing fold; Exclude last tokens in every sentence!
PRC: 0.6001, REC: 0.6088, F1: 0.6044; MAE: 0.1237;

#### Attempt 7, Правило 4 доработано: Длинное слово (от 10 символов) с высокой плотностью согласных
Calculate metrics, training fold; Exclude last tokens in every sentence!
PRC: 0.6244, REC: 0.6106, F1: 0.6174; MAE: 0.1265;

Calculate metrics, testing fold; Exclude last tokens in every sentence!
PRC: 0.6159, REC: 0.6063, F1: 0.6110; MAE: 0.1240;

#### Attempt 8, Правило 4 доработано: Длинное слово (от 11 символов) с высокой плотностью согласных
Calculate metrics, training fold; Exclude last tokens in every sentence!
PRC: 0.6345, REC: 0.6032, F1: 0.6185; MAE: 0.1267;

Calculate metrics, testing fold; Exclude last tokens in every sentence!
PRC: 0.6269, REC: 0.5985, F1: 0.6124; MAE: 0.1240;

#### Attempt 9, Правило 4 доработано: Длинное слово (от 12 символов) с высокой плотностью согласных, замена доли гласных на 33%
Calculate metrics, training fold; Exclude last tokens in every sentence!
PRC: 0.6792, REC: 0.5804, F1: 0.6259; MAE: 0.1269;

Calculate metrics, testing fold; Exclude last tokens in every sentence!
PRC: 0.6735, REC: 0.5760, F1: 0.6210; MAE: 0.1235;

#### Attempt 10, В проверку знаков препинания добавила многоточие
Calculate metrics, training fold; Exclude last tokens in every sentence!
PRC: 0.6937, REC: 0.6303, F1: 0.6604; MAE: 0.1268;

Calculate metrics, testing fold; Exclude last tokens in every sentence!
PRC: 0.6908, REC: 0.6328, F1: 0.6605; MAE: 0.1245;

#### Attempt 11, пауза перед союзом И
Calculate metrics, training fold; Exclude last tokens in every sentence!
PRC: 0.6728, REC: 0.6564, F1: 0.6645; MAE: 0.1258;

Calculate metrics, testing fold; Exclude last tokens in every sentence!
PRC: 0.6696, REC: 0.6575, F1: 0.6635; MAE: 0.1236;

#### Attempt 12, пауза перед словом "авторов"
Calculate metrics, training fold; Exclude last tokens in every sentence!
PRC: 0.6730, REC: 0.6569, F1: 0.6649; MAE: 0.1258;

Calculate metrics, testing fold; Exclude last tokens in every sentence!
PRC: 0.6698, REC: 0.6581, F1: 0.6639; MAE: 0.1235;

#### Attempt 13, после некоторых вводных слов не предсказываем паузу (очевидно, наверное ...)
Calculate metrics, training fold; Exclude last tokens in every sentence!
PRC: 0.6741, REC: 0.6563, F1: 0.6651; MAE: 0.1258;

Calculate metrics, testing fold; Exclude last tokens in every sentence!
PRC: 0.6714, REC: 0.6576, F1: 0.6644; MAE: 0.1235;

In [26]:
%run pause_predictor.py

Running predictions...


100%|██████████████████████████████████████████████████████████████████████████| 21327/21327 [00:02<00:00, 8046.91it/s]


Calculate metrics, training fold; Exclude last tokens in every sentence!
PRC: 0.6742, REC: 0.6564, F1: 0.6651; MAE: 0.1258;

Calculate metrics, testing fold; Exclude last tokens in every sentence!
PRC: 0.6714, REC: 0.6576, F1: 0.6644; MAE: 0.1235;
